In [1]:
from pathlib import Path
import json

PROJECT_ROOT = Path(
    "/home/sagemaker-user/AAI-540-Project/"
    "multilingual-it-ticket-routing-mlops"
)

config_path = PROJECT_ROOT / "project_config.json"
assert config_path.is_file(), "project_config.json was not found."

project_config = json.loads(config_path.read_text())

AWS_REGION = project_config["aws_region"]
PROJECT_BUCKET = project_config["s3_bucket"]

deployment = project_config["deployment"]
quality_report = project_config["monitoring"]["production_quality"]

assert deployment["status"] == "Completed", (
    "The saved batch deployment is not marked Completed."
)

print("Region:", AWS_REGION)
print("Project bucket:", PROJECT_BUCKET)
print("Completed batch job:", deployment["job_name"])
print("Production quality report:", quality_report["uri"])
print("\nMonitoring setup ready.")

Region: us-east-1
Project bucket: aai540-group3-tickets-390568313988-us-east-1
Completed batch job: aai540-group3-batch-20261004-045734
Production quality report: s3://aai540-group3-tickets-390568313988-us-east-1/monitoring/aai540-group3-batch-20261004-045734/production_quality.json

Monitoring setup ready.


In [2]:
from io import BytesIO
from urllib.parse import urlparse

import boto3
import pandas as pd

aws_session = boto3.Session(region_name=AWS_REGION)
s3_client = aws_session.client("s3")


def read_versioned_csv(file_info):
    location = urlparse(file_info["uri"])
    version_id = file_info["version_id"]

    assert version_id and version_id != "null", "Missing S3 version ID."

    response = s3_client.get_object(
        Bucket=location.netloc,
        Key=location.path.lstrip("/"),
        VersionId=version_id,
    )

    assert response["VersionId"] == version_id

    with response["Body"] as body:
        return pd.read_csv(
            BytesIO(body.read()),
            dtype={"ticket_id": "string"},
            low_memory=False,
        )


training_df = read_versioned_csv(project_config["s3_data"]["train"])
production_df = read_versioned_csv(
    project_config["s3_data"]["production"]
)

required_inputs = {"ticket_id", "subject", "body", "language"}

for name, frame in [
    ("Training", training_df),
    ("Production", production_df),
]:
    assert required_inputs.issubset(frame.columns)
    assert frame["ticket_id"].notna().all()
    assert frame["ticket_id"].is_unique
    print(f"{name}: {len(frame):,} tickets loaded")

assert len(training_df) == 17710
assert len(production_df) == 17712
assert "queue" not in production_df.columns
assert set(training_df["ticket_id"]).isdisjoint(
    set(production_df["ticket_id"])
)

print("\nVerified: separate datasets and no production queue labels.")
print("Ready to build the training-data reference.")

Training: 17,710 tickets loaded
Production: 17,712 tickets loaded

Verified: separate datasets and no production queue labels.
Ready to build the training-data reference.


In [3]:
import numpy as np


def monitoring_features(frame):
    subject = frame["subject"].fillna("").astype(str).str.strip()
    body = frame["body"].fillna("").astype(str).str.strip()
    text = (subject + " " + body).str.strip()

    language = (
        frame["language"]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .str.lower()
        .replace("", "unknown")
    )

    return pd.DataFrame({
        "language": language,
        "text_char_count": text.str.len(),
        "text_word_count": text.str.split().str.len(),
        "missing_subject": subject.eq(""),
        "missing_body": body.eq(""),
        "empty_text": text.eq(""),
        "unknown_language": language.eq("unknown"),
    }, index=frame.index)


training_monitor_features = monitoring_features(training_df)

training_baseline = {
    "reference_split": "train",
    "source": project_config["s3_data"]["train"],
    "rows": len(training_df),
    "language_proportions": (
        training_monitor_features["language"]
        .value_counts(normalize=True)
        .sort_index()
        .to_dict()
    ),
    "missing_rates": {},
    "numeric_features": {},
}

for column in [
    "missing_subject", "missing_body", "empty_text", "unknown_language"
]:
    training_baseline["missing_rates"][column] = float(
        training_monitor_features[column].mean()
    )

for column in ["text_char_count", "text_word_count"]:
    values = training_monitor_features[column].to_numpy()

    # Define bins using training data only.
    # The first and last bins also cover future values outside its range.
    cut_points = np.unique(
        np.quantile(values, np.arange(0.1, 1.0, 0.1))
    )
    bin_indices = np.searchsorted(cut_points, values, side="right")
    bin_counts = np.bincount(
        bin_indices, minlength=len(cut_points) + 1
    )

    training_baseline["numeric_features"][column] = {
        "mean": float(values.mean()),
        "median": float(np.median(values)),
        "p95": float(np.quantile(values, 0.95)),
        "cut_points": cut_points.tolist(),
        "bin_proportions": (bin_counts / len(values)).tolist(),
    }

monitoring_report_dir = PROJECT_ROOT / "reports" / "monitoring"
monitoring_report_dir.mkdir(parents=True, exist_ok=True)

baseline_path = monitoring_report_dir / "training_data_baseline.json"
baseline_path.write_text(
    json.dumps(training_baseline, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Reference tickets:", training_baseline["rows"])
print("Languages:", sorted(training_baseline["language_proportions"]))
print("Missing-text rates:", training_baseline["missing_rates"])
print("Training reference saved:", baseline_path)

Reference tickets: 17710
Languages: ['de', 'en', 'es', 'fr', 'pt']
Missing-text rates: {'missing_subject': 0.10316205533596838, 'missing_body': 0.0, 'empty_text': 0.0, 'unknown_language': 0.0}
Training reference saved: /home/sagemaker-user/AAI-540-Project/multilingual-it-ticket-routing-mlops/reports/monitoring/training_data_baseline.json


In [4]:
# Set thresholds before calculating production monitoring results.
data_thresholds = {
    "distribution_distance": 0.10,
    "missing_rate_increase": 0.05,
    "empty_text_rate": 0.0,
    "unknown_language_rate": 0.0,
    "unseen_language_rate": 0.0,
}

production_monitor_features = monitoring_features(production_df)
data_checks = []


def add_check(name, value, threshold):
    data_checks.append({
        "check": name,
        "value": float(value),
        "threshold": float(threshold),
        "status": "ALERT" if value > threshold else "OK",
    })


# Total variation distance: 0 means identical distributions;
# 1 means completely non-overlapping distributions.
reference_languages = training_baseline["language_proportions"]
current_languages = (
    production_monitor_features["language"]
    .value_counts(normalize=True)
    .to_dict()
)

all_languages = sorted(
    set(reference_languages) | set(current_languages)
)

language_distance = 0.5 * sum(
    abs(
        reference_languages.get(language, 0.0)
        - current_languages.get(language, 0.0)
    )
    for language in all_languages
)

add_check(
    "language_distribution",
    language_distance,
    data_thresholds["distribution_distance"],
)

# Reuse the training bins; do not create new bins from production.
for column, reference in training_baseline["numeric_features"].items():
    values = production_monitor_features[column].to_numpy()
    bin_indices = np.searchsorted(
        reference["cut_points"], values, side="right"
    )
    counts = np.bincount(
        bin_indices,
        minlength=len(reference["bin_proportions"]),
    )
    proportions = counts / len(values)

    distance = 0.5 * np.abs(
        proportions - np.array(reference["bin_proportions"])
    ).sum()

    add_check(
        f"{column}_distribution",
        distance,
        data_thresholds["distribution_distance"],
    )

for column in ["missing_subject", "missing_body"]:
    increase = max(
        0.0,
        float(production_monitor_features[column].mean())
        - training_baseline["missing_rates"][column],
    )
    add_check(
        f"{column}_rate_increase",
        increase,
        data_thresholds["missing_rate_increase"],
    )

for column in ["empty_text", "unknown_language"]:
    add_check(
        f"{column}_rate",
        production_monitor_features[column].mean(),
        data_thresholds[f"{column}_rate"],
    )

unseen_language_rate = (
    ~production_monitor_features["language"].isin(reference_languages)
).mean()

add_check(
    "unseen_language_rate",
    unseen_language_rate,
    data_thresholds["unseen_language_rate"],
)

data_quality_report = {
    "reference_source": training_baseline["source"],
    "production_source": project_config["s3_data"]["production"],
    "production_rows": len(production_df),
    "thresholds": data_thresholds,
    "checks": data_checks,
    "alert_count": sum(item["status"] == "ALERT" for item in data_checks),
}

(monitoring_report_dir / "production_data_quality.json").write_text(
    json.dumps(data_quality_report, indent=2, allow_nan=False),
    encoding="utf-8",
)

display(pd.DataFrame(data_checks).round(4))
print("Alerts:", data_quality_report["alert_count"])
print("Production data monitoring report saved.")

,check,value,threshold,status
0,language_distribution,0.0042,0.10,OK
1,text_char_count_distribution,0.0127,0.10,OK
2,text_word_count_distribution,0.0103,0.10,OK
3,missing_subject_rate_increase,0.0008,0.05,OK
4,missing_body_rate_increase,0.0001,0.05,OK
5,empty_text_rate,0.0000,0.00,OK
6,unknown_language_rate,0.0000,0.00,OK
7,unseen_language_rate,0.0000,0.00,OK


Alerts: 0
Production data monitoring report saved.


In [5]:
# Load the saved production quality report from its exact S3 version.
report_location = urlparse(quality_report["uri"])

response = s3_client.get_object(
    Bucket=report_location.netloc,
    Key=report_location.path.lstrip("/"),
    VersionId=quality_report["version_id"],
)

assert response["VersionId"] == quality_report["version_id"]

with response["Body"] as body:
    production_quality = json.loads(body.read())

assert production_quality["batch_job_name"] == deployment["job_name"]

current_metrics = production_quality["metrics"]
reference_metrics = project_config["final_model"]["test_metrics"]

assert current_metrics["tickets"] == len(production_df)

# Initial demonstration threshold: an absolute drop of 0.05.
allowed_drop = 0.05
model_checks = []

for metric in ["accuracy", "macro_f1", "weighted_f1"]:
    reference_value = float(reference_metrics[metric])
    current_value = float(current_metrics[metric])

    assert 0.0 <= reference_value <= 1.0
    assert 0.0 <= current_value <= 1.0

    minimum_value = max(0.0, reference_value - allowed_drop)

    model_checks.append({
        "metric": metric,
        "test_reference": reference_value,
        "production_value": current_value,
        "minimum_allowed": minimum_value,
        "status": "ALERT" if current_value < minimum_value else "OK",
    })

model_quality_report = {
    "batch_job_name": deployment["job_name"],
    "training_job_name": project_config["final_model"]["training_job_name"],
    "reference_split": "test",
    "reference_source": project_config["final_model"]["test_dataset"],
    "production_report_source": {
        "uri": quality_report["uri"],
        "version_id": quality_report["version_id"],
    },
    "allowed_absolute_drop": allowed_drop,
    "checks": model_checks,
    "alert_count": sum(
        item["status"] == "ALERT" for item in model_checks
    ),
}

(monitoring_report_dir / "model_quality_checks.json").write_text(
    json.dumps(model_quality_report, indent=2, allow_nan=False),
    encoding="utf-8",
)

display(pd.DataFrame(model_checks).round(4))
print("Model quality alerts:", model_quality_report["alert_count"])
print("Model quality monitoring report saved.")

,metric,test_reference,production_value,minimum_allowed,status
0,accuracy,0.4634,0.4591,0.4134,OK
1,macro_f1,0.4464,0.4380,0.3964,OK
2,weighted_f1,0.4653,0.4605,0.4153,OK


Model quality alerts: 0
Model quality monitoring report saved.


In [6]:
from datetime import datetime, timezone

sagemaker_client = aws_session.client("sagemaker")

training_details = sagemaker_client.describe_training_job(
    TrainingJobName=project_config["final_model"]["training_job_name"]
)

batch_details = sagemaker_client.describe_transform_job(
    TransformJobName=deployment["job_name"]
)


def summarize_job(details, job_type):
    if job_type == "training":
        name = details["TrainingJobName"]
        status = details["TrainingJobStatus"]
        start = details.get("TrainingStartTime")
        end = details.get("TrainingEndTime")
    else:
        name = details["TransformJobName"]
        status = details["TransformJobStatus"]
        start = details.get("TransformStartTime")
        end = details.get("TransformEndTime")

    duration = (
        (end - start).total_seconds()
        if start is not None and end is not None
        else None
    )

    if status == "Completed":
        check_status = "OK"
    elif status in {"Failed", "Stopped", "Stopping"}:
        check_status = "ALERT"
    else:
        check_status = "PENDING"

    return {
        "job_type": job_type,
        "job_name": name,
        "aws_status": status,
        "check_status": check_status,
        "duration_seconds": duration,
        "start_time": start.isoformat() if start else None,
        "end_time": end.isoformat() if end else None,
        "failure_reason": details.get("FailureReason"),
    }


infrastructure_jobs = [
    summarize_job(training_details, "training"),
    summarize_job(batch_details, "batch_prediction"),
]

infrastructure_report = {
    "checked_at": datetime.now(timezone.utc).isoformat(),
    "jobs": infrastructure_jobs,
    "alert_count": sum(
        job["check_status"] == "ALERT" for job in infrastructure_jobs
    ),
}

(monitoring_report_dir / "infrastructure_status.json").write_text(
    json.dumps(infrastructure_report, indent=2, allow_nan=False),
    encoding="utf-8",
)

display(pd.DataFrame(infrastructure_jobs)[[
    "job_type",
    "aws_status",
    "duration_seconds",
    "check_status",
]])

print("Infrastructure alerts:", infrastructure_report["alert_count"])
print("Infrastructure status report saved.")

,job_type,aws_status,duration_seconds,check_status
0,training,Completed,118.783,OK
1,batch_prediction,Completed,72.528,OK


Infrastructure alerts: 0
Infrastructure status report saved.


In [7]:
cloudwatch_client = aws_session.client("cloudwatch")

METRIC_NAMESPACE = "AAI540/TicketRouting"
metric_dimensions = [
    {"Name": "Project", "Value": "aai540-group3"},
    {"Name": "Environment", "Value": "simulated-production"},
]

snapshot_time = datetime.now(timezone.utc)
metric_data = []


def add_metric(name, value, unit="None"):
    metric_data.append({
        "MetricName": name,
        "Dimensions": metric_dimensions,
        "Timestamp": snapshot_time,
        "Value": float(value),
        "Unit": unit,
    })


# Model performance: values remain on a 0–1 scale.
for metric, name in [
    ("accuracy", "ProductionAccuracy"),
    ("macro_f1", "ProductionMacroF1"),
    ("weighted_f1", "ProductionWeightedF1"),
]:
    add_metric(name, current_metrics[metric])

# Number of checks that triggered alerts.
add_metric("DataAlertCount", data_quality_report["alert_count"], "Count")
add_metric(
    "ModelQualityAlertCount", model_quality_report["alert_count"], "Count"
)
add_metric(
    "InfrastructureAlertCount",
    infrastructure_report["alert_count"],
    "Count",
)

# Individual input-data checks.
data_metric_names = {
    "language_distribution": "LanguageDistributionDistance",
    "text_char_count_distribution": "TextCharDistributionDistance",
    "text_word_count_distribution": "TextWordDistributionDistance",
    "missing_subject_rate_increase": "MissingSubjectRateIncrease",
    "missing_body_rate_increase": "MissingBodyRateIncrease",
    "empty_text_rate": "EmptyTextRate",
    "unknown_language_rate": "UnknownLanguageRate",
    "unseen_language_rate": "UnseenLanguageRate",
}

for check in data_checks:
    add_metric(data_metric_names[check["check"]], check["value"])

# Job runtimes and status checks.
for job in infrastructure_jobs:
    prefix = "Training" if job["job_type"] == "training" else "Batch"

    if job["duration_seconds"] is not None:
        add_metric(
            f"{prefix}DurationSeconds",
            job["duration_seconds"],
            "Seconds",
        )

    add_metric(
        f"{prefix}JobAlert",
        int(job["check_status"] == "ALERT"),
        "Count",
    )

result = cloudwatch_client.put_metric_data(
    Namespace=METRIC_NAMESPACE,
    MetricData=metric_data,
)

assert result["ResponseMetadata"]["HTTPStatusCode"] == 200

project_config["monitoring"]["cloudwatch"] = {
    "namespace": METRIC_NAMESPACE,
    "dimensions": metric_dimensions,
    "last_published_at": snapshot_time.isoformat(),
    "batch_job_name": deployment["job_name"],
    "metric_names": [item["MetricName"] for item in metric_data],
}

config_path.write_text(
    json.dumps(project_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("CloudWatch accepted metrics:", len(metric_data))
print("Namespace:", METRIC_NAMESPACE)
print("Monitoring configuration saved.")

CloudWatch accepted metrics: 18
Namespace: AAI540/TicketRouting
Monitoring configuration saved.


In [8]:
dashboard_name = "AAI540-Group3-Ticket-Monitoring"

# Each dashboard metric must use the same dimensions as publication.
dimension_fields = [
    value
    for dimension in metric_dimensions
    for value in (dimension["Name"], dimension["Value"])
]


def metric_widget(title, metric_names, x, y, width=12):
    return {
        "type": "metric",
        "x": x,
        "y": y,
        "width": width,
        "height": 6,
        "properties": {
            "title": title,
            "region": AWS_REGION,
            "view": "singleValue",
            "sparkline": False,
            "stat": "Average",
            "period": 300,
            "metrics": [
                [METRIC_NAMESPACE, name, *dimension_fields]
                for name in metric_names
            ],
        },
    }


dashboard_body = {
    "start": "-P7D",
    "periodOverride": "inherit",
    "widgets": [
        {
            "type": "text",
            "x": 0,
            "y": 0,
            "width": 24,
            "height": 4,
            "properties": {
                "markdown": (
                    "# Multilingual ticket routing — monitoring\n"
                    f"Batch job: `{deployment['job_name']}`\n\n"
                    f"Snapshot published: `{snapshot_time.isoformat()}`\n\n"
                    "Metrics are published when monitoring runs. "
                    "This is not yet scheduled monitoring. "
                    "Quality scores and data rates use a 0–1 scale. "
                    "Zero alerts means the configured checks passed; "
                    "it does not establish readiness for automatic routing."
                )
            },
        },
        metric_widget(
            "Production model quality — higher is better",
            [
                "ProductionAccuracy",
                "ProductionMacroF1",
                "ProductionWeightedF1",
            ],
            0, 4,
        ),
        metric_widget(
            "Triggered checks — zero means no alerts",
            [
                "DataAlertCount",
                "ModelQualityAlertCount",
                "InfrastructureAlertCount",
            ],
            12, 4,
        ),
        metric_widget(
            "Data distribution changes — alert above 0.10",
            [
                "LanguageDistributionDistance",
                "TextCharDistributionDistance",
                "TextWordDistributionDistance",
            ],
            0, 10,
        ),
        metric_widget(
            "AWS job duration — seconds",
            ["TrainingDurationSeconds", "BatchDurationSeconds"],
            12, 10,
        ),
        metric_widget(
            "Input quality — missing-rate increases and invalid-input rates",
            [
                "MissingSubjectRateIncrease",
                "MissingBodyRateIncrease",
                "EmptyTextRate",
                "UnknownLanguageRate",
                "UnseenLanguageRate",
            ],
            0, 16, width=24,
        ),
    ],
}

dashboard_json = json.dumps(dashboard_body, indent=2)

(monitoring_report_dir / "cloudwatch_dashboard.json").write_text(
    dashboard_json, encoding="utf-8"
)

result = cloudwatch_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=dashboard_json,
)

validation_messages = result.get("DashboardValidationMessages", [])
if validation_messages:
    raise RuntimeError(
        "Dashboard validation messages: "
        + json.dumps(validation_messages, indent=2)
    )

dashboard_url = (
    f"https://console.aws.amazon.com/cloudwatch/home?region={AWS_REGION}"
    f"#dashboards:name={dashboard_name}"
)

project_config["monitoring"]["cloudwatch"].update({
    "dashboard_name": dashboard_name,
    "dashboard_url": dashboard_url,
})

config_path.write_text(
    json.dumps(project_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Dashboard created:", dashboard_name)
print("Open this link while signed into your AWS lab:")
print(dashboard_url)

Dashboard created: AAI540-Group3-Ticket-Monitoring
Open this link while signed into your AWS lab:
https://console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=AAI540-Group3-Ticket-Monitoring


In [9]:
from datetime import timedelta

published_at = datetime.fromisoformat(
    project_config["monitoring"]["cloudwatch"]["last_published_at"]
)

metric_check = cloudwatch_client.get_metric_statistics(
    Namespace=METRIC_NAMESPACE,
    MetricName="ProductionAccuracy",
    Dimensions=metric_dimensions,
    StartTime=published_at - timedelta(minutes=5),
    EndTime=published_at + timedelta(minutes=5),
    Period=60,
    Statistics=["Average"],
)

points = sorted(
    metric_check["Datapoints"],
    key=lambda point: point["Timestamp"],
)

print("Accuracy data points found:", len(points))

for point in points:
    print(
        "Timestamp:", point["Timestamp"].isoformat(),
        "| Accuracy:", round(point["Average"], 4),
    )

Accuracy data points found: 1
Timestamp: 2026-10-04T05:32:00+00:00 | Accuracy: 0.4591


In [10]:
# Pin the dashboard to the saved monitoring snapshot.
window_start = published_at - timedelta(minutes=5)
window_end = published_at + timedelta(minutes=5)

start_text = window_start.strftime("%Y-%m-%dT%H:%M:%SZ")
end_text = window_end.strftime("%Y-%m-%dT%H:%M:%SZ")

dashboard_body["start"] = start_text
dashboard_body["end"] = end_text

for widget in dashboard_body["widgets"]:
    if widget["type"] == "metric":
        widget["properties"].update({
            "start": start_text,
            "end": end_text,
            "setPeriodToTimeRange": True,
            "sparkline": False,
        })

dashboard_json = json.dumps(dashboard_body, indent=2)

result = cloudwatch_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=dashboard_json,
)

messages = result.get("DashboardValidationMessages", [])
if messages:
    raise RuntimeError(json.dumps(messages, indent=2))

(monitoring_report_dir / "cloudwatch_dashboard.json").write_text(
    dashboard_json, encoding="utf-8"
)

project_config["monitoring"]["cloudwatch"].update({
    "dashboard_window_start": start_text,
    "dashboard_window_end": end_text,
})

config_path.write_text(
    json.dumps(project_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Dashboard updated to show the saved batch snapshot.")
print(dashboard_url)

Dashboard updated to show the saved batch snapshot.
https://console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=AAI540-Group3-Ticket-Monitoring


In [11]:
alarm_definitions = {
    "AAI540-Group3-DataQuality": "DataAlertCount",
    "AAI540-Group3-ModelQuality": "ModelQualityAlertCount",
    "AAI540-Group3-Infrastructure": "InfrastructureAlertCount",
}

for alarm_name, metric_name in alarm_definitions.items():
    cloudwatch_client.put_metric_alarm(
        AlarmName=alarm_name,
        AlarmDescription=(
            f"Flag when {metric_name} exceeds zero. "
            "Metrics are published by the ticket monitoring workflow."
        ),
        Namespace=METRIC_NAMESPACE,
        MetricName=metric_name,
        Dimensions=metric_dimensions,
        Statistic="Maximum",
        Period=3600,
        EvaluationPeriods=1,
        DatapointsToAlarm=1,
        Threshold=0.0,
        ComparisonOperator="GreaterThanThreshold",
        TreatMissingData="missing",
        ActionsEnabled=False,
    )
    print("Alarm created:", alarm_name)

alarm_response = cloudwatch_client.describe_alarms(
    AlarmNames=list(alarm_definitions)
)

alarm_records = [
    {
        "name": alarm["AlarmName"],
        "arn": alarm["AlarmArn"],
        "metric": alarm["MetricName"],
        "state": alarm["StateValue"],
    }
    for alarm in alarm_response["MetricAlarms"]
]

assert len(alarm_records) == 3

project_config["monitoring"]["cloudwatch"]["alarms"] = alarm_records

config_path.write_text(
    json.dumps(project_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

display(pd.DataFrame(alarm_records)[["name", "metric", "state"]])
print("Alarm configuration saved.")

Alarm created: AAI540-Group3-DataQuality
Alarm created: AAI540-Group3-ModelQuality
Alarm created: AAI540-Group3-Infrastructure


,name,metric,state
0,AAI540-Group3-DataQuality,DataAlertCount,INSUFFICIENT_DATA
1,AAI540-Group3-Infrastructure,InfrastructureAlertCount,INSUFFICIENT_DATA
2,AAI540-Group3-ModelQuality,ModelQualityAlertCount,INSUFFICIENT_DATA


Alarm configuration saved.


In [12]:
import hashlib

# Refresh the alarm states without recreating the alarms.
alarm_response = cloudwatch_client.describe_alarms(
    AlarmNames=list(alarm_definitions)
)

alarm_records = [
    {
        "name": alarm["AlarmName"],
        "arn": alarm["AlarmArn"],
        "metric": alarm["MetricName"],
        "state": alarm["StateValue"],
    }
    for alarm in alarm_response["MetricAlarms"]
]

alarm_report = {
    "checked_at": datetime.now(timezone.utc).isoformat(),
    "alarms": alarm_records,
}

(monitoring_report_dir / "alarm_status.json").write_text(
    json.dumps(alarm_report, indent=2),
    encoding="utf-8",
)

report_names = [
    "training_data_baseline.json",
    "production_data_quality.json",
    "model_quality_checks.json",
    "infrastructure_status.json",
    "cloudwatch_dashboard.json",
    "alarm_status.json",
]

saved_artifacts = {}

for filename in report_names:
    report_path = monitoring_report_dir / filename
    report_bytes = report_path.read_bytes()
    object_key = f"monitoring/{deployment['job_name']}/{filename}"

    upload = s3_client.put_object(
        Bucket=PROJECT_BUCKET,
        Key=object_key,
        Body=report_bytes,
        ContentType="application/json",
    )

    version_id = upload.get("VersionId")
    assert version_id and version_id != "null"

    saved_artifacts[report_path.stem] = {
        "uri": f"s3://{PROJECT_BUCKET}/{object_key}",
        "version_id": version_id,
        "sha256": hashlib.sha256(report_bytes).hexdigest(),
    }

    print("Saved to S3:", filename)

monitoring_config = project_config["monitoring"]
monitoring_config.setdefault("artifacts", {}).update(saved_artifacts)
monitoring_config["data_thresholds"] = data_thresholds
monitoring_config["model_allowed_absolute_drop"] = allowed_drop
monitoring_config["cloudwatch"]["alarms"] = alarm_records

config_path.write_text(
    json.dumps(project_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

display(pd.DataFrame(alarm_records)[["name", "state"]])
print("Versioned monitoring reports and configuration saved.")

Saved to S3: training_data_baseline.json
Saved to S3: production_data_quality.json
Saved to S3: model_quality_checks.json
Saved to S3: infrastructure_status.json
Saved to S3: cloudwatch_dashboard.json
Saved to S3: alarm_status.json


,name,state
0,AAI540-Group3-DataQuality,INSUFFICIENT_DATA
1,AAI540-Group3-Infrastructure,OK
2,AAI540-Group3-ModelQuality,INSUFFICIENT_DATA


Versioned monitoring reports and configuration saved.
